# Cateorical

In [33]:
import pandas as pd 
VINs = pd.Series(['AX193Q43', 'Z11RTV201', 'WA4Q3371', 'QWP77491'], dtype='string')

print(VINs)


0     AX193Q43
1    Z11RTV201
2     WA4Q3371
3     QWP77491
dtype: string


In [34]:
colours = pd.Categorical(values=[ 'red','blue','red','green'])
colours
#you can see there are 3 catergries 
#order is asecending
#this is not series by you make it by wrapping in series 

['red', 'blue', 'red', 'green']
Categories (3, object): ['blue', 'green', 'red']

In [35]:
pd.Series(colours)
#pandas sets the values as non-unique nan values

0      red
1     blue
2      red
3    green
dtype: category
Categories (3, object): ['blue', 'green', 'red']

In [36]:
colors = pd.Categorical(
    values = ['red', 'blue', 'red', 'green'],
    categories = ['black', 'blue', 'green', 'orange', 'red', 'yellow']
)
print(colors)

['red', 'blue', 'red', 'green']
Categories (6, object): ['black', 'blue', 'green', 'orange', 'red', 'yellow']


In [37]:
sizes = pd.Categorical(
    values=['standard', 'mini', 'standard', 'extended'],
    categories = ['mini', 'standard', 'extended'],
    ordered=True
)
sizes

['standard', 'mini', 'standard', 'extended']
Categories (3, object): ['mini' < 'standard' < 'extended']

In [38]:
sizes< 'extended'

array([ True,  True,  True, False])

In [39]:
sizesSeries = pd.Series(
    data = [0,1,2,3],
    index = pd.CategoricalIndex(sizes)
)
sizesSeries



standard    0
mini        1
standard    2
extended    3
dtype: int64

you can do one hot encding for machine learning input 

In [40]:
pd.get_dummies(sizes, prefix = 'size') 

,size_mini,size_standard,size_extended
0,0,1,0
1,1,0,0
2,0,1,0
3,0,0,1


# Mutiindexing

create multiindex

In [41]:
store_products = pd.DataFrame(
    data = {'Price': [35.25, 45.00, 23.50, 1.95, 29.99, 35.65]},
    index = pd.MultiIndex.from_tuples([
        ('super store', 'basketball'), 
        ('super store', 'football'), 
        ('super store', 'soccerball'),

        ('sports dorks', 'golfball'),
        ('sports dorks', 'basketball'), 
        ('sports dorks', 'football')

    ], names=['store', 'product'])
)

print(store_products)
#understand it have multiple level of rows and columns 

                         Price
store        product          
super store  basketball  35.25
             football    45.00
             soccerball  23.50
sports dorks golfball     1.95
             basketball  29.99
             football    35.65


In [42]:
#    level 0      level 1
#  _ _ _ _ _ _  _ _ _ _ _ _  Price                       
# |store       | product   |       
# |super store | basketball|  35.25
# |            | football  |  45.00
# |            | soccerball|  23.50
# |sports dorks| golfball  |   1.95
# |_ _ _ _ _ _ | basketball|  29.99
#              | football  |  35.65  
#              |_ _ _ _ _ _| 

back to normal index

In [43]:
store_products.reset_index(inplace=True)
store_products

,store,product,Price
0,super store,basketball,35.25
1,super store,football,45.00
2,super store,soccerball,23.50
3,sports dorks,golfball,1.95
4,sports dorks,basketball,29.99
5,sports dorks,football,35.65


make to multiindex

In [44]:
store_products.set_index(['store', 'product'], inplace=True)
print(store_products)

                         Price
store        product          
super store  basketball  35.25
             football    45.00
             soccerball  23.50
sports dorks golfball     1.95
             basketball  29.99
             football    35.65


indexing 

In [45]:
store_products.loc[[('sports dorks','basketball'),('super store','football')]]

,,Price
store,product,
sports dorks,basketball,29.99
super store,football,45.00


In [46]:
# if you want to slecet footbal not easily
#store_products.loc['football']

In [47]:
store_products.xs(key = 'football', level='product')

,Price
store,
super store,45.00
sports dorks,35.65


In [48]:
store_products.xs(key = 'football', level='product',drop_level=False)

,,Price
store,product,
super store,football,45.00
sports dorks,football,35.65


Groupby

In [49]:
#multiindex occurs in groupby sometimes
df = pd.DataFrame({
    'A': ['foo', 'bar', 'foo', 'bar', 'bar', 'foo', 'foo'],
    'B': [False, True, False, True, True, True, True],
    'C': [2.1, 1.9, 3.6, 4.0, 1.9, 7.8, 2.8],
    'D': [50, 30, 30, 90, 10, 20, 10]
})
display(df)

,A,B,C,D
0,foo,False,2.1,50
1,bar,True,1.9,30
2,foo,False,3.6,30
3,bar,True,4.0,90
4,bar,True,1.9,10
5,foo,True,7.8,20
6,foo,True,2.8,10


In [67]:
stew = df.groupby(by= ['A','B']).agg({'C': ['sum'], 'D': ['sum', 'mean']})
display(stew)

C    D           
            sum  sum       mean
A   B                          
bar True    7.8  130  43.333333
foo False   5.7   80  40.000000
    True   10.6   30  15.000000

In [51]:
stew.columns

MultiIndex([('C',  'sum'),
            ('D',  'sum'),
            ('D', 'mean')],
           )

In [52]:
stew.index

MultiIndex([('bar',  True),
            ('foo', False),
            ('foo',  True)],
           names=['A', 'B'])

In [68]:
stew.xs(
    key='sum',
    axis= 1,
    level= 1,
    drop_level= False
)


C    D
            sum  sum
A   B               
bar True    7.8  130
foo False   5.7   80
    True   10.6   30

In [54]:
stew.xs(
        key='D',
        axis=1,
        level=0,
        drop_level=False
    )

D           
           sum       mean
A   B                    
bar True   130  43.333333
foo False   80  40.000000
    True    30  15.000000

In [69]:
stew.columns.to_flat_index()

Index([('C', 'sum'), ('D', 'sum'), ('D', 'mean')], dtype='object')

In [71]:
['_'.join(s) for s in stew.columns.to_flat_index()]

['C_sum', 'D_sum', 'D_mean']

In [72]:
stew.columns =['_'.join(s) for s in stew.columns.to_flat_index()]

print(stew)

           C_sum  D_sum     D_mean
A   B                             
bar True     7.8    130  43.333333
foo False    5.7     80  40.000000
    True    10.6     30  15.000000


# Pivot

In [74]:
df1 = pd.DataFrame({
    'row': ['row0', 'row1', 'row2', 'row0', 'row1', 'row2'],
    'col': ['col1', 'col1', 'col1', 'col0', 'col0', 'col0'],
    'val': [44, 47, 64, 67, 67,  9]
})
display(df1)


,row,col,val
0,row0,col1,44
1,row1,col1,47
2,row2,col1,64
3,row0,col0,67
4,row1,col0,67
5,row2,col0,9


In [78]:
# we convert long format to wide format
df1.pivot(
    index = 'row',
    columns= 'col',
    values= 'val'
)

col,col0,col1
row,,
row0,67,44
row1,67,47
row2,9,64


# pivot table
what happen if you have multiple entries in dataframe so use pivot table

In [79]:
df2 = pd.DataFrame({
    'row': ['row0', 'row0', 'row2', 'row2', 'row1', 'row1'],
    'col': ['col1', 'col1', 'col1', 'col0', 'col0', 'col0'],
    'val': [44, 47, 64, 67, 67, 9]
})
display(df2)


,row,col,val
0,row0,col1,44
1,row0,col1,47
2,row2,col1,64
3,row2,col0,67
4,row1,col0,67
5,row1,col0,9


In [81]:
df2.pivot_table(
    index = 'row',
    columns= 'col',
    values= 'val',
    aggfunc=list
)

col,col0,col1
row,,
row0,NaN,"[44, 47]"
row1,"[67, 9]",NaN
row2,[67],[64]


In [91]:
df2.pivot_table(
    index = 'row',
    columns= 'col',
    values= 'val',
    aggfunc=list,
    fill_value=0,
    margins= True,
)

col,col0,col1,All
row,,,
row0,0,"[44, 47]","[44, 47]"
row1,"[67, 9]",0,"[67, 9]"
row2,[67],[64],"[64, 67]"
All,"[67, 67, 9]","[44, 47, 64]","[44, 47, 64, 67, 67, 9]"


In [92]:
toomuch = pd.DataFrame({
    'row_major': ['A', 'A', 'B', 'A', 'B', 'B'],
    'row_minor': ['x', 'x', 'y', 'y', 'z', 'z'],
    'col_major': ['MAMMAL', 'MAMMAL', 'MAMMAL', 'FISH', 'FISH', 'FISH'],
    'col_minor': ['dog', 'cat', 'dog', 'tuna', 'tuna', 'shark'],
    'val0': [44, 47, 64, 67, 67,  9],
    'val1': [91, 52, 86, 83, 79, 92]
})
display(toomuch)

,row_major,row_minor,col_major,col_minor,val0,val1
0,A,x,MAMMAL,dog,44,91
1,A,x,MAMMAL,cat,47,52
2,B,y,MAMMAL,dog,64,86
3,A,y,FISH,tuna,67,83
4,B,z,FISH,tuna,67,79
5,B,z,FISH,shark,9,92


example

In [97]:
pv = toomuch.pivot_table(
    index=['row_major', 'row_minor'],
    columns=['col_major', 'col_minor'],
    values=['val0', 'val1'],
    aggfunc=['count', 'sum']
)
pv

count                                           sum        \
                     val0                   val1                   val0         
col_major            FISH      MAMMAL       FISH      MAMMAL       FISH         
col_minor           shark tuna    cat  dog shark tuna    cat  dog shark  tuna   
row_major row_minor                                                             
A         x           NaN  NaN    1.0  1.0   NaN  NaN    1.0  1.0   NaN   NaN   
          y           NaN  1.0    NaN  NaN   NaN  1.0    NaN  NaN   NaN  67.0   
B         y           NaN  NaN    NaN  1.0   NaN  NaN    NaN  1.0   NaN   NaN   
          z           1.0  1.0    NaN  NaN   1.0  1.0    NaN  NaN   9.0  67.0   

                                                           
                                  val1                     
col_major           MAMMAL        FISH       MAMMAL        
col_minor              cat   dog shark  tuna    cat   dog  
row_major row_minor                                        
A         x           47.0  44.0   NaN   NaN   52.0  91.0  
          y            NaN   NaN   NaN  83.0    NaN   NaN  
B         y            NaN  64.0   NaN   NaN    NaN  86.0  
          z            NaN   NaN  92.0  79.0    NaN   NaN

In [98]:
#see it have four levels of columns
pv.columns

MultiIndex([('count', 'val0',   'FISH', 'shark'),
            ('count', 'val0',   'FISH',  'tuna'),
            ('count', 'val0', 'MAMMAL',   'cat'),
            ('count', 'val0', 'MAMMAL',   'dog'),
            ('count', 'val1',   'FISH', 'shark'),
            ('count', 'val1',   'FISH',  'tuna'),
            ('count', 'val1', 'MAMMAL',   'cat'),
            ('count', 'val1', 'MAMMAL',   'dog'),
            (  'sum', 'val0',   'FISH', 'shark'),
            (  'sum', 'val0',   'FISH',  'tuna'),
            (  'sum', 'val0', 'MAMMAL',   'cat'),
            (  'sum', 'val0', 'MAMMAL',   'dog'),
            (  'sum', 'val1',   'FISH', 'shark'),
            (  'sum', 'val1',   'FISH',  'tuna'),
            (  'sum', 'val1', 'MAMMAL',   'cat'),
            (  'sum', 'val1', 'MAMMAL',   'dog')],
           names=[None, None, 'col_major', 'col_minor'])

# melt
wide table -> long table

In [100]:
wide = pd.DataFrame({
    'test': [1, 2, 3, 4],
    'john': [95, 81, 47, 99],
    'patty': [90, 85, 93, 97]
})
display(wide)

,test,john,patty
0,1,95,90
1,2,81,85
2,3,47,93
3,4,99,97


In [101]:
wide.melt(
    id_vars='test',
    value_vars=['john','patty'],
    var_name='student',
    value_name='score'
)

,test,student,score
0,1,john,95
1,2,john,81
2,3,john,47
3,4,john,99
4,1,patty,90
5,2,patty,85
6,3,patty,93
7,4,patty,97


In [102]:
wide.melt(value_vars=['john','patty'])

,variable,value
0,john,95
1,john,81
2,john,47
3,john,99
4,patty,90
5,patty,85
6,patty,93
7,patty,97


# stack and unstack